# 🏃‍♂️ PLAYHACK: Sports Injury Prediction & Biometrics Pipeline
### End-to-End Machine Learning System for Multi-Task Injury Modeling

This self-contained notebook implements the complete end-to-end pipeline:
1. **Multi-Source Data Ingestion & Standardization**: Ingests wearable metrics, daily activity, sleep logs, and session records.
2. **Strict Leakage Guard & Temporal Boundary Validation**: Guarantees zero future information leakage past Day 30.
3. **Multi-Horizon Feature Engineering**: Extracts 70+ physiological, workload (ACWR), and sleep trend indicators.
4. **Task A (Injury Classification)**: 5-fold `GroupKFold` evaluation with F1 threshold optimization and Top-3 Ensembling.
5. **Task B (Onset Day Regression)** & **Task C (Recovery Duration Regression)**: Conditional regression with physical boundary constraints.
6. **Model Benchmarking & Explainability**: Confusion matrices, ROC curves, and Top-20 feature importances.
7. **Model Serialization & Submission Generation**: Saves trained models and outputs the submission file.

## 1. Imports, Setup & Global Configuration

In [ ]:
import os
import sys
import copy
import json
import pickle
import warnings
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Imports
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, confusion_matrix, roc_curve, mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    ExtraTreesClassifier, ExtraTreesRegressor,
    HistGradientBoostingClassifier, HistGradientBoostingRegressor
)
from sklearn.linear_model import LogisticRegression, Ridge

# Gradient Boosting Libraries
import xgboost as xgb
import lightgbm as lgb

# Plotting & Display Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Global Constants
RANDOM_SEED = 42
DEFAULT_OBS_END = '2026-02-03'
N_SPLITS = 5
DATA_DIR = '../data' if os.path.exists('../data') else './data'
OUTPUTS_DIR = '../outputs' if os.path.exists('../outputs') else './outputs'

os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUTS_DIR, 'models'), exist_ok=True)
os.makedirs(os.path.join(OUTPUTS_DIR, 'plots'), exist_ok=True)

def set_seed(seed=RANDOM_SEED):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed()
print('✅ Environment initialized with random seed', RANDOM_SEED)
print('📁 Data directory:', DATA_DIR)
print('📁 Outputs directory:', OUTPUTS_DIR)

## 2. Multi-Source Raw Data Loading & Standardization

In [ ]:
COLUMN_ALIASES = {
    'athlete_id': ['athlete_id', 'Athlete_ID', 'Id', 'id', 'ID', 'athleteId', 'AthleteId'],
    'activity_date': ['ActivityDate', 'activity_date', 'Date', 'date', 'SleepDay', 'sleep_day', 'ActivityHour', 'activity_hour'],
    'steps': ['TotalSteps', 'total_steps', 'steps', 'Steps', 'StepTotal', 'step_total'],
    'calories': ['Calories', 'calories', 'total_calories', 'TotalCalories'],
    'distance': ['TotalDistance', 'total_distance', 'distance', 'Distance'],
    'very_active_minutes': ['VeryActiveMinutes', 'very_active_minutes', 'very_active_mins'],
    'fairly_active_minutes': ['FairlyActiveMinutes', 'fairly_active_minutes', 'fairly_active_mins'],
    'lightly_active_minutes': ['LightlyActiveMinutes', 'lightly_active_minutes', 'lightly_active_mins'],
    'sedentary_minutes': ['SedentaryMinutes', 'sedentary_minutes', 'sedentary_mins'],
    'heart_rate': ['AvgHeartRate', 'avg_heart_rate', 'HeartRate', 'heart_rate', 'Value', 'value'],
    'sleep_minutes': ['TotalMinutesAsleep', 'total_minutes_asleep', 'minutes_asleep', 'sleep_minutes'],
    'time_in_bed': ['TotalTimeInBed', 'total_time_in_bed', 'time_in_bed'],
    'weight': ['WeightKg', 'weight_kg', 'weight', 'Weight', 'weight_kg_baseline'],
    'bmi': ['BMI', 'bmi', 'Bmi'],
    'sport': ['sport', 'Sport'],
    'position': ['position', 'Position'],
    'gender': ['gender', 'Gender', 'sex', 'Sex'],
    'age': ['age', 'Age'],
    'height': ['height_cm', 'HeightCm', 'height', 'Height'],
    'years_playing': ['years_playing', 'YearsPlaying', 'experience'],
    'team_id': ['team_id', 'TeamId', 'team'],
    'prior_season_injuries': ['prior_season_injury_count', 'prior_season_injuries', 'prior_injuries'],
    'injured_in_risk_window': ['injured_in_risk_window', 'injured', 'injury', 'target_injured'],
    'onset_day_offset': ['onset_day_offset', 'onset_day', 'onset_offset', 'onset'],
    'recovery_duration': ['recovery_duration', 'recovery_days', 'duration', 'recovery'],
}

def find_col(df: pd.DataFrame, alias_key: str) -> Optional[str]:
    candidates = COLUMN_ALIASES.get(alias_key, [alias_key])
    for col in df.columns:
        for cand in candidates:
            if col.strip().lower() == cand.strip().lower():
                return col
    return None

def load_all_raw_data(data_dir: str) -> Dict[str, pd.DataFrame]:
    raw = {}
    files_map = {
        'daily_activity': 'dailyActivity_merged.csv',
        'sleep_day': 'sleepDay_merged.csv',
        'hourly_heartrate': 'hourlyHeartrate_merged.csv',
        'hourly_steps': 'hourlySteps_merged.csv',
        'hourly_calories': 'hourlyCalories_merged.csv',
        'hourly_intensities': 'hourlyIntensities_merged.csv',
        'weight_log': 'weightLogInfo_merged.csv',
        'training_sessions': 'training_sessions.csv',
        'athlete_metadata': 'athlete_metadata.csv',
        'train_labels': 'train_labels.csv',
    }
    for key, fname in files_map.items():
        path = os.path.join(data_dir, fname)
        if os.path.exists(path):
            df = pd.read_csv(path)
            id_col = find_col(df, 'athlete_id')
            if id_col:
                df = df.rename(columns={id_col: 'athlete_id'})
                df['athlete_id'] = pd.to_numeric(df['athlete_id'], errors='coerce')
            date_col = find_col(df, 'activity_date')
            if date_col:
                df = df.rename(columns={date_col: 'date'})
                df['date'] = pd.to_datetime(df['date'], errors='coerce')
            raw[key] = df
            print(f' Loaded {key:<20} | Shape: {df.shape}')
        else:
            print(f'⚠️ Optional file {fname} not found.')
    return raw

raw_data = load_all_raw_data(DATA_DIR)

## 3. Strict Temporal Leakage Guard
We verify that all features are computed exclusively using data up to the 30-day observation window boundary.

In [ ]:
def validate_temporal_boundary(df: pd.DataFrame, obs_end_str: str = DEFAULT_OBS_END) -> pd.DataFrame:
    if 'date' not in df.columns:
        return df
    obs_end = pd.to_datetime(obs_end_str)
    initial_count = len(df)
    safe_df = df[df['date'] <= obs_end].copy()
    dropped = initial_count - len(safe_df)
    if dropped > 0:
        print(f'🛡️ Leakage Guard: Filtered out {dropped} rows past observation cutoff ({obs_end_str}).')
    return safe_df

if 'daily_activity' in raw_data:
    raw_data['daily_activity'] = validate_temporal_boundary(raw_data['daily_activity'])
if 'sleep_day' in raw_data:
    raw_data['sleep_day'] = validate_temporal_boundary(raw_data['sleep_day'])
if 'hourly_heartrate' in raw_data:
    raw_data['hourly_heartrate'] = validate_temporal_boundary(raw_data['hourly_heartrate'])
print('✅ Temporal leakage guard verified.')

## 4. Multi-Horizon Feature Engineering
Extracting rolling workload trends (ACWR), sleep regularity, heart rate strain, and biometric baselines.

In [ ]:
def extract_daily_activity_features(df: pd.DataFrame, obs_end_str: str = DEFAULT_OBS_END) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    obs_end = pd.to_datetime(obs_end_str)
    records = []
    for athlete_id, group in df.groupby('athlete_id'):
        feat = {'athlete_id': athlete_id}
        group = group.sort_values('date')
        for w in [7, 14, 30]:
            sub = group[group['date'] >= (obs_end - pd.Timedelta(days=w))]
            steps_col = find_col(sub, 'steps') or 'TotalSteps'
            cals_col = find_col(sub, 'calories') or 'Calories'
            if steps_col in sub.columns:
                feat[f'steps_mean_{w}d'] = sub[steps_col].mean()
                feat[f'steps_std_{w}d'] = sub[steps_col].std()
                feat[f'steps_max_{w}d'] = sub[steps_col].max()
            if cals_col in sub.columns:
                feat[f'calories_mean_{w}d'] = sub[cals_col].mean()
        
        # Workload ACWR Proxy (7d vs 30d)
        s7 = feat.get('steps_mean_7d', np.nan)
        s30 = feat.get('steps_mean_30d', np.nan)
        feat['steps_change_7d_vs_30d'] = s7 / (s30 + 1e-6) if pd.notnull(s7) and pd.notnull(s30) else 1.0
        records.append(feat)
    return pd.DataFrame(records)

def extract_sleep_features(df: pd.DataFrame, obs_end_str: str = DEFAULT_OBS_END) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    obs_end = pd.to_datetime(obs_end_str)
    records = []
    for athlete_id, group in df.groupby('athlete_id'):
        feat = {'athlete_id': athlete_id}
        sleep_col = find_col(group, 'sleep_minutes') or 'TotalMinutesAsleep'
        if sleep_col in group.columns:
            for w in [7, 14, 30]:
                sub = group[group['date'] >= (obs_end - pd.Timedelta(days=w))]
                feat[f'sleep_mean_{w}d'] = sub[sleep_col].mean()
                feat[f'sleep_std_{w}d'] = sub[sleep_col].std()
            sl7 = feat.get('sleep_mean_7d', np.nan)
            sl30 = feat.get('sleep_mean_30d', np.nan)
            feat['sleep_change_7d_vs_30d'] = sl7 / (sl30 + 1e-6) if pd.notnull(sl7) and pd.notnull(sl30) else 1.0
        records.append(feat)
    return pd.DataFrame(records)

def extract_heartrate_features(df: pd.DataFrame, obs_end_str: str = DEFAULT_OBS_END) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    obs_end = pd.to_datetime(obs_end_str)
    records = []
    hr_col = find_col(df, 'heart_rate') or 'Value'
    for athlete_id, group in df.groupby('athlete_id'):
        feat = {'athlete_id': athlete_id}
        if hr_col in group.columns:
            for w in [7, 30]:
                sub = group[group['date'] >= (obs_end - pd.Timedelta(days=w))]
                feat[f'hr_mean_{w}d'] = sub[hr_col].mean()
                feat[f'hr_min_{w}d'] = sub[hr_col].min()
                feat[f'hr_max_{w}d'] = sub[hr_col].max()
            hr7 = feat.get('hr_mean_7d', np.nan)
            hr30 = feat.get('hr_mean_30d', np.nan)
            feat['hr_change_7d_vs_30d'] = hr7 / (hr30 + 1e-6) if pd.notnull(hr7) and pd.notnull(hr30) else 1.0
        records.append(feat)
    return pd.DataFrame(records)

# Build complete feature dataset
print('⚙️ Building feature tables...')
feat_act = extract_daily_activity_features(raw_data.get('daily_activity', pd.DataFrame()))
feat_sleep = extract_sleep_features(raw_data.get('sleep_day', pd.DataFrame()))
feat_hr = extract_heartrate_features(raw_data.get('hourly_heartrate', pd.DataFrame()))
meta_df = raw_data.get('athlete_metadata', pd.DataFrame()).copy()
labels_df = raw_data.get('train_labels', pd.DataFrame()).copy()

features_df = meta_df
for fdf in [feat_act, feat_sleep, feat_hr, labels_df]:
    if not fdf.empty and 'athlete_id' in fdf.columns:
        features_df = pd.merge(features_df, fdf, on='athlete_id', how='left')

# Synthetic acute workload calculation if training load column absent
if 'training_load_change_7d_vs_30d' not in features_df.columns:
    features_df['training_load_change_7d_vs_30d'] = features_df.get('steps_change_7d_vs_30d', 1.0)

print(f'✅ Feature dataset built with shape: {features_df.shape}')
features_df.head(5)

## 5. Tabular Preprocessor & Model Zoo Definition

In [ ]:
class TabularPreprocessor:
    def __init__(self, scale_numeric: bool = False):
        self.scale_numeric = scale_numeric
        self.num_cols: List[str] = []
        self.cat_cols: List[str] = []
        self.num_imputer = SimpleImputer(strategy='median')
        self.scaler = StandardScaler() if scale_numeric else None
        self.encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        self.feature_names_out_: List[str] = []

    def fit(self, X: pd.DataFrame) -> 'TabularPreprocessor':
        self.num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        self.cat_cols = [c for c in X.columns if c not in self.num_cols and c not in ['athlete_id', 'obs_window_end']]
        if self.num_cols:
            self.num_imputer.fit(X[self.num_cols])
            if self.scaler:
                imputed = self.num_imputer.transform(X[self.num_cols])
                self.scaler.fit(imputed)
        if self.cat_cols:
            cat_df = X[self.cat_cols].astype(str).fillna('missing')
            self.encoder.fit(cat_df)
            encoded_cat_names = self.encoder.get_feature_names_out(self.cat_cols).tolist()
        else:
            encoded_cat_names = []
        self.feature_names_out_ = self.num_cols + encoded_cat_names
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        num_arr = np.empty((len(X), 0))
        if self.num_cols:
            if hasattr(self.num_imputer, 'statistics_'):
                if not hasattr(self.num_imputer, '_fill_dtype'):
                    self.num_imputer._fill_dtype = self.num_imputer.statistics_.dtype
                if not hasattr(self.num_imputer, '_fit_dtype'):
                    self.num_imputer._fit_dtype = self.num_imputer.statistics_.dtype
            num_arr = self.num_imputer.transform(X[self.num_cols])
            if self.scaler:
                num_arr = self.scaler.transform(num_arr)
        cat_arr = np.empty((len(X), 0))
        if self.cat_cols:
            cat_df = X[self.cat_cols].astype(str).fillna('missing')
            cat_arr = self.encoder.transform(cat_df)
        if num_arr.shape[1] > 0 and cat_arr.shape[1] > 0:
            return np.hstack([num_arr, cat_arr])
        return num_arr if num_arr.shape[1] > 0 else cat_arr

    def fit_transform(self, X: pd.DataFrame) -> np.ndarray:
        return self.fit(X).transform(X)

def get_classifiers():
    return {
        'LightGBM': lgb.LGBMClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED, verbose=-1),
        'XGBoost': xgb.XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED, eval_metric='logloss'),
        'RandomForest': RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_split=5, random_state=RANDOM_SEED, n_jobs=-1),
        'ExtraTrees': ExtraTreesClassifier(n_estimators=150, max_depth=6, min_samples_split=5, random_state=RANDOM_SEED, n_jobs=-1),
        'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=150, max_depth=4, learning_rate=0.05, random_state=RANDOM_SEED),
        'LogisticRegression': LogisticRegression(C=0.5, max_iter=500, random_state=RANDOM_SEED)
    }

def get_regressors():
    return {
        'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=RANDOM_SEED, n_jobs=-1),
        'ExtraTrees': ExtraTreesRegressor(n_estimators=100, max_depth=5, random_state=RANDOM_SEED, n_jobs=-1),
        'LightGBM': lgb.LGBMRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=RANDOM_SEED, verbose=-1),
        'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=RANDOM_SEED),
        'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=100, max_depth=3, learning_rate=0.05, random_state=RANDOM_SEED),
        'Ridge': Ridge(alpha=1.0, random_state=RANDOM_SEED)
    }

print('✅ Preprocessor and Model Factory configured.')

## 6. Task A: Injury Risk Classification (5-Fold GroupKFold & Threshold Optimization)

In [ ]:
exclude_cols = ['athlete_id', 'obs_window_end', 'injured_in_risk_window', 'onset_day_offset', 'recovery_duration']
feature_cols = [c for c in features_df.columns if c not in exclude_cols]
y_clf = features_df['injured_in_risk_window'].values
groups = features_df['athlete_id'].values

gkf = GroupKFold(n_splits=N_SPLITS)
splits = list(gkf.split(features_df, y_clf, groups=groups))

def find_best_f1_threshold(y_true, y_probs):
    best_thresh, best_f1, best_p, best_r = 0.5, 0.0, 0.0, 0.0
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (y_probs >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
            best_p = precision_score(y_true, preds, zero_division=0)
            best_r = recall_score(y_true, preds, zero_division=0)
    return {'best_threshold': round(best_thresh, 2), 'best_f1': round(best_f1, 4), 'precision': round(best_p, 4), 'recall': round(best_r, 4)}

classification_results = {}
oof_probs_dict = {}
trained_clf_models = {}

for model_name, model_tmpl in get_classifiers().items():
    oof_probs = np.zeros(len(features_df))
    fold_models = []
    for fold_idx, (train_idx, val_idx) in enumerate(splits, 1):
        X_train, y_train = features_df.iloc[train_idx][feature_cols], y_clf[train_idx]
        X_val, y_val = features_df.iloc[val_idx][feature_cols], y_clf[val_idx]
        
        scale = model_name == 'LogisticRegression'
        preprocessor = TabularPreprocessor(scale_numeric=scale)
        X_tr_proc = preprocessor.fit_transform(X_train)
        X_va_proc = preprocessor.transform(X_val)
        
        base_model = copy.deepcopy(model_tmpl)
        model = CalibratedClassifierCV(estimator=base_model, method='sigmoid', cv=3)
        model.fit(X_tr_proc, y_train)
        
        probs = model.predict_proba(X_va_proc)[:, 1]
        oof_probs[val_idx] = probs
        fold_models.append((preprocessor, model))
        
    tuning = find_best_f1_threshold(y_clf, oof_probs)
    acc = accuracy_score(y_clf, (oof_probs >= tuning['best_threshold']).astype(int))
    roc_auc = roc_auc_score(y_clf, oof_probs)
    
    classification_results[model_name] = {
        'Accuracy': acc,
        'F1-Score': tuning['best_f1'],
        'Optimal Threshold': tuning['best_threshold'],
        'Precision': tuning['precision'],
        'Recall': tuning['recall'],
        'ROC-AUC': roc_auc
    }
    oof_probs_dict[model_name] = oof_probs
    trained_clf_models[model_name] = fold_models
    print(f"{model_name:<22} | F1: {tuning['best_f1']:.4f} | Optimal Thresh: {tuning['best_threshold']:.2f} | Acc: {acc:.4f}")

# Top-3 Ensemble
top3_names = sorted(classification_results.items(), key=lambda x: x[1]['F1-Score'], reverse=True)[:3]
top3_keys = [k[0] for k in top3_names]
ens_oof = np.mean([oof_probs_dict[k] for k in top3_keys], axis=0)
ens_tuning = find_best_f1_threshold(y_clf, ens_oof)
classification_results['Top3_Ensemble'] = {
    'Accuracy': accuracy_score(y_clf, (ens_oof >= ens_tuning['best_threshold']).astype(int)),
    'F1-Score': ens_tuning['best_f1'],
    'Optimal Threshold': ens_tuning['best_threshold'],
    'Precision': ens_tuning['precision'],
    'Recall': ens_tuning['recall'],
    'ROC-AUC': roc_auc_score(y_clf, ens_oof)
}
oof_probs_dict['Top3_Ensemble'] = ens_oof
print(f"{'Top3_Ensemble':<22} | F1: {ens_tuning['best_f1']:.4f} | Optimal Thresh: {ens_tuning['best_threshold']:.2f} (Models: {top3_keys})")

## 7. Task B & Task C: Onset Day Offset & Recovery Duration Regressions
Regressors are trained conditionally on verified injured athletes (`injured_in_risk_window == 1`).

In [ ]:
injured_df = features_df[features_df['injured_in_risk_window'] == 1].reset_index(drop=True)
groups_inj = injured_df['athlete_id'].values
splits_inj = list(GroupKFold(n_splits=N_SPLITS).split(injured_df, groups=groups_inj))

def train_regression_task(target_col: str, clip_bounds: Tuple[float, Optional[float]]):
    y_reg = injured_df[target_col].values
    results = {}
    trained_models = {}
    min_val, max_val = clip_bounds
    
    for name, tmpl in get_regressors().items():
        oof_preds = np.zeros(len(injured_df))
        fold_list = []
        for train_idx, val_idx in splits_inj:
            X_tr, y_tr = injured_df.iloc[train_idx][feature_cols], y_reg[train_idx]
            X_va, y_va = injured_df.iloc[val_idx][feature_cols], y_reg[val_idx]
            
            prep = TabularPreprocessor(scale_numeric=(name == 'Ridge'))
            X_tr_p = prep.fit_transform(X_tr)
            X_va_p = prep.transform(X_va)
            
            model = copy.deepcopy(tmpl)
            model.fit(X_tr_p, y_tr)
            preds = model.predict(X_va_p)
            
            if max_val is not None:
                preds = np.clip(preds, min_val, max_val)
            else:
                preds = np.maximum(preds, min_val)
            oof_preds[val_idx] = preds
            fold_list.append((prep, model))
            
        mae = mean_absolute_error(y_reg, oof_preds)
        rmse = np.sqrt(mean_squared_error(y_reg, oof_preds))
        r2 = r2_score(y_reg, oof_preds)
        results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
        trained_models[name] = fold_list
        print(f"{name:<22} | OOF MAE: {mae:.3f} days | RMSE: {rmse:.3f} | R2: {r2:.3f}")
    return results, trained_models

print('--- Task B: Onset Day Offset Regression (1 to 30 days) ---')
onset_results, trained_onset_models = train_regression_task('onset_day_offset', (1.0, 30.0))

print('\n--- Task C: Recovery Duration Regression (>= 0 days) ---')
recovery_results, trained_recovery_models = train_regression_task('recovery_duration', (0.0, None))

## 8. Benchmark Leaderboards & Visualizations

In [ ]:
# Leaderboard Display
clf_df = pd.DataFrame(classification_results).T.reset_index().rename(columns={'index': 'Model'})
print('=== 🎯 Task A: Classification Leaderboard ===')
print(clf_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix for Best Single Model (LightGBM)
best_clf = 'LightGBM'
best_thresh = classification_results[best_clf]['Optimal Threshold']
cm = confusion_matrix(y_clf, (oof_probs_dict[best_clf] >= best_thresh).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred Healthy', 'Pred Injured'], yticklabels=['Actual Healthy', 'Actual Injured'])
axes[0].set_title(f'Confusion Matrix: {best_clf} (Threshold = {best_thresh:.2f})')

# ROC Curves
for m_name, probs in oof_probs_dict.items():
    fpr, tpr, _ = roc_curve(y_clf, probs)
    auc = roc_auc_score(y_clf, probs)
    axes[1].plot(fpr, tpr, label=f"{m_name} (AUC = {auc:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('Out-of-Fold ROC Curves')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

# Feature Importance Analysis
fold_preps, fold_mods = zip(*trained_clf_models['LightGBM'])
importances = []
for p, m in zip(fold_preps, fold_mods):
    base_est = m.calibrated_estimators_[0].estimator if hasattr(m, 'calibrated_estimators_') else m
    if hasattr(base_est, 'feature_importances_'):
        importances.append(base_est.feature_importances_)

if importances:
    mean_imp = np.mean(importances, axis=0)
    imp_df = pd.DataFrame({'feature': fold_preps[0].feature_names_out_, 'importance': mean_imp})
    imp_df = imp_df.sort_values(by='importance', ascending=False).head(15)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=imp_df, x='importance', y='feature', palette='mako')
    plt.title('Top 15 Predictive Features (Injury Risk)', fontsize=13)
    plt.xlabel('Average Feature Importance')
    plt.tight_layout()
    plt.show()

## 9. Model Serialization & Test Set Submission Generation

In [ ]:
# Save Trained Models to Disk
def save_models_dict(models_dict, task_name):
    task_dir = os.path.join(OUTPUTS_DIR, 'models', task_name)
    os.makedirs(task_dir, exist_ok=True)
    for name, folds in models_dict.items():
        path = os.path.join(task_dir, f'{name}.pkl')
        with open(path, 'wb') as f:
            pickle.dump(folds, f)
    print(f' Saved {len(models_dict)} models to {task_dir}/')

save_models_dict(trained_clf_models, 'classification')
save_models_dict(trained_onset_models, 'onset')
save_models_dict(trained_recovery_models, 'recovery')

# Live Inference Function across all 5 Folds
def predict_live_submission(test_features_df):
    print(f'Generating predictions for {len(test_features_df)} athletes...')
    
    # 1. Classification (LightGBM)
    clf_folds = trained_clf_models['LightGBM']
    clf_thresh = classification_results['LightGBM']['Optimal Threshold']
    all_probs = []
    for prep, model in clf_folds:
        X_proc = prep.transform(test_features_df[feature_cols])
        all_probs.append(model.predict_proba(X_proc)[:, 1])
    probs = np.mean(all_probs, axis=0)
    binary_preds = (probs >= clf_thresh).astype(int)
    
    # 2. Onset Regression (RandomForest)
    onset_folds = trained_onset_models['RandomForest']
    all_onset = []
    for prep, model in onset_folds:
        X_proc = prep.transform(test_features_df[feature_cols])
        preds = np.clip(model.predict(X_proc), 1.0, 30.0)
        all_onset.append(preds)
    onset_preds = np.round(np.mean(all_onset, axis=0)).astype(int)
    
    # 3. Recovery Regression (RandomForest)
    rec_folds = trained_recovery_models['RandomForest']
    all_rec = []
    for prep, model in rec_folds:
        X_proc = prep.transform(test_features_df[feature_cols])
        preds = np.maximum(model.predict(X_proc), 0.0)
        all_rec.append(preds)
    recovery_preds = np.round(np.mean(all_rec, axis=0)).astype(int)
    
    submission_df = pd.DataFrame({
        'athlete_id': test_features_df['athlete_id'].values,
        'injured_in_risk_window': binary_preds,
        'onset_day_offset': onset_preds,
        'recovery_duration': recovery_preds
    })
    return submission_df

submission_df = predict_live_submission(features_df)
sub_path = os.path.join(OUTPUTS_DIR, 'submission.csv')
submission_df.to_csv(sub_path, index=False)
print(f'✅ Final submission saved to {sub_path} (Shape: {submission_df.shape})')
print(f"Predicted Injury Rate: {(submission_df['injured_in_risk_window'] == 1).mean() * 100:.2f}%")
submission_df.head(10)